In [81]:
import numpy as np 
import pandas as pd 

import re
import ssl
from urllib import request
import pymorphy3
import matplotlib.pyplot as plt
import seaborn as sns 

from catboost import CatBoostRegressor, Pool
import catboost as cb
from catboost import MetricVisualizer

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold

from scipy.sparse import csr_matrix
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
import spacy
from functools import lru_cache
from wordcloud import WordCloud
from tqdm.auto import trange

import os

from typing import Optional, Literal
from collections import Counter

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/maksimdegtarev/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [82]:
# Чтение данных
train = pd.read_parquet("train.parquet")
test = pd.read_parquet("test.parquet")
train.head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
0,185689,2024-09-27,Б/у,3000.0,Транспорт,Запчасти и аксессуары,Салон,1942218,1935418,Ручка АКПП mercedes w203 avangarde,Ручка рычаг акпп на Мерседес В203 mercedes w20...,185689.jpg,0.370,10.0,23.0,19.0
1,1914373,2024-11-07,Новое с биркой,5990.0,Личные вещи,"Одежда, обувь, аксессуары",Зимние куртки и пуховики,2164034,1753243,Пуховик Moncler голубой (52 размер),Объявление для заказа 📲\n\nАвито доставка 🚚\n\...,1914373.jpg,2.486,14.0,37.0,24.0
2,361626,2024-12-15,Новое,1200.0,Транспорт,Запчасти и аксессуары,Двигатель,621511,1233378,Запчасти на ford фокус1,"Опора задняя,двигатель w тигуан 1,4 150 б.у",361626.jpg,0.640,7.0,23.0,18.0
3,534927,2024-01-20,Б/у,13000.0,Электроника,"Игры, приставки и программы",Игровые приставки и аксессуары,998450,1082324,Ps3 cechc 08 скальпирована HEN(полностью испра...,"В комплекте приставка, провода(зарядка, питани...",534927.jpg,7.100,20.0,35.0,20.0
4,199043,2024-07-28,Отличное,300.0,Личные вещи,"Одежда, обувь, аксессуары","Джемперы, свитеры, кардиганы",528098,477834,Свитер трикотаж 44 р-р Reserved,Свитер Pull&Bear (пул энд бир)\n44 размер\nНад...,199043.jpg,0.400,7.0,23.0,11.0


### 1. Обработка пропусков

In [83]:
def get_missed_values_stat(df):
    missed_stat = df.isna().sum().sort_values(ascending=False).reset_index()
    missed_stat.columns = ['feature', 'NaN count']
    missed_stat['NaN share'] = missed_stat['NaN count'] / df.shape[0]
    return missed_stat[missed_stat['NaN count'] > 0]

def get_common_missed_data(df_train, df_test):
    missed_train = get_missed_values_stat(df_train)
    missed_test = get_missed_values_stat(df_test)

    missed_data = missed_train.merge(missed_test, how='outer', on='feature', suffixes=['_train', '_test'])
    
    return missed_data

In [84]:
# Проведем обработку пропусков
missed_data = get_common_missed_data(train, test)
missed_data

,feature,NaN count_train,NaN share_train,NaN count_test,NaN share_test
0,item_condition,16294,0.052073,3835,0.054572


#### Заменим пропуски в item_condition на "unknown"

In [85]:

train["item_condition"] = train["item_condition"].fillna("unknown").astype("str")
test["item_condition"] = test["item_condition"].fillna("unknown").astype("str")
missed_data = get_common_missed_data(train, test)
missed_data

,feature,NaN count_train,NaN share_train,NaN count_test,NaN share_test


In [86]:
seed = 42

# случайный сплит 90/10
train, val = train_test_split(
    train,
    test_size=0.1,
    random_state=seed,
    shuffle=True
)
print(f"train_size: {train.shape[0]}, val_size: {val.shape[0]}")

train_size: 281617, val_size: 31291


### 2. Первичный анализ данных (EDA) 

#### Анализ выбросов

In [87]:
targets = ["real_height", "real_length", "real_width", "real_weight"]
train[targets].describe().T

,count,mean,std,min,25%,50%,75%,max
real_height,281617.0,11.620215,8.325536,1.000,6.00,10.00,15.0,288.0
real_length,281617.0,33.191100,215.065406,1.000,22.00,32.00,40.0,111363.0
real_width,281617.0,23.502587,23.376426,1.000,16.00,22.00,30.0,11029.0
real_weight,281617.0,1.707787,20.997758,0.001,0.34,0.69,1.5,7090.0


#### Анализ таргетов указывает на наличие выбросов, подозрительно большие значения максимумов, особенно для real_length

In [88]:
train[targets].quantile([0.95, 0.99, 0.995, 0.999]).T

,0.950,0.990,0.995,0.999
real_height,29.0,36.0000,40.0,51.000000
real_length,56.0,99.0000,116.0,150.000000
real_width,44.0,52.0000,60.0,87.000000
real_weight,6.0,14.3084,18.0,24.416128


#### Квантили демонстрируют границы значительно меньшие чем максимальные значения, что подтвверждает предположение что объявления с максимальными значениями - выбросы. Рассмотрим примеры из данных, которые выходят за 0.999 квантиль

In [89]:
train[train["real_height"] > train["real_height"].quantile(0.999)].sort_values(by="real_height", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
95915,1675102,2024-08-19,Б/у,100.0,Для дома и дачи,Посуда и товары для кухни,Товары для кухни,2178921,2391516,Хоз товары и тряпки,"Хоз товары, тряпки",1675102.jpg,15.81,288.0,437.0,321.0
34669,1088814,2024-03-28,Б/у,800.0,Электроника,Фототехника,Оборудование и аксессуары,422671,1949683,Цифровая фоторамка view sonic,"Модель VFМ1036w-11Е\nВ хорошем состоянии, почт...",1088814.jpg,1.81,153.0,333.0,253.0
197162,1661424,2024-11-12,Б/у,5400.0,Электроника,Оргтехника и расходники,"МФУ, копиры и сканеры",1093475,30370,Мфу лазерный Ricoh sp220 snw,"Хорошее рабочее состояние, без дефектов! Новый...",1661424.jpg,38.22,135.0,162.0,138.0
249173,1875634,2024-10-27,Б/у,9500.0,Для дома и дачи,Бытовая техника,Очистители воздуха,1153747,267046,Мойка воздуха Venta lw15,Увлажнитель - мойка воздуха venta lw15. В отли...,1875634.jpg,11.07,108.0,150.0,120.0
21117,1834023,2024-11-04,Новое с биркой,500.0,Личные вещи,"Одежда, обувь, аксессуары",Рюкзаки,1697859,626647,Новый классный рюкзак Гуси.Черный.Вышлю почтой,продам новый рюкзак Гуси.Вышлю почтой.,1834023.jpg,6.00,105.0,150.0,105.0


In [90]:
train[train["real_length"] > train["real_length"].quantile(0.999)].sort_values(by="real_length", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
37315,1099165,2024-05-28,Б/у,2000.0,Электроника,Фототехника,Зеркальные фотоаппараты,1145987,891526,Тушка Canon 1000d,"Тушка canon 1000d, работает отлично, состояние...",1099165.jpg,0.850,16.0,111363.0,28.0
180033,216891,2024-01-11,unknown,370.0,Личные вещи,Красота и здоровье,Средства для волос,1345411,1467631,Schwarzkopf professional Кондиционер для волос,Кондиционер для волос PEPTIDE REPAIR rescue дл...,216891.jpg,0.560,1.0,21789.0,11029.0
77757,1686069,2024-08-23,Хорошее,30.0,Личные вещи,"Одежда, обувь, аксессуары",Сумки,534652,1523587,Отправлено Большая сумка женской обуви,"Продаю свою обувь: сапоги ботильоны, туфли и т...",1686069.jpg,12.900,30.0,7010.0,33.0
5264,618796,2024-01-21,Б/у,635.0,Хобби и отдых,Книги и журналы,Учебная литература,26525,1750451,"Учебник английского языка 1, 2, 3 класс технол...","Продаются учебники английского языка для 1, 2 ...",618796.jpg,1.075,3.0,3023.0,23.0
144014,683384,2024-01-13,Б/у,300.0,Хобби и отдых,Книги и журналы,Книги,492168,286503,"Кролик, который хочет уснуть",Книга в идеальным состоянии.\n\nВ профиле есть...,683384.jpg,0.476,6.0,2015.0,10.0


In [91]:
train[train["real_width"] > train["real_width"].quantile(0.999)].sort_values(by="real_width", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
180033,216891,2024-01-11,unknown,370.0,Личные вещи,Красота и здоровье,Средства для волос,1345411,1467631,Schwarzkopf professional Кондиционер для волос,Кондиционер для волос PEPTIDE REPAIR rescue дл...,216891.jpg,0.56,1.0,21789.0,11029.0
95915,1675102,2024-08-19,Б/у,100.0,Для дома и дачи,Посуда и товары для кухни,Товары для кухни,2178921,2391516,Хоз товары и тряпки,"Хоз товары, тряпки",1675102.jpg,15.81,288.0,437.0,321.0
34669,1088814,2024-03-28,Б/у,800.0,Электроника,Фототехника,Оборудование и аксессуары,422671,1949683,Цифровая фоторамка view sonic,"Модель VFМ1036w-11Е\nВ хорошем состоянии, почт...",1088814.jpg,1.81,153.0,333.0,253.0
240024,1836483,2024-10-23,Б/у,5000.0,Личные вещи,Товары для детей и игрушки,Куклы и аксессуары,538886,348581,Дом мечты барби,"В идеальном состоянии, в комплекте куча мебели...",1836483.jpg,39.00,66.0,225.0,225.0
307037,1745944,2024-10-26,Б/у,540.0,Личные вещи,Товары для детей и игрушки,Постельные принадлежности,1400307,934415,Матрас для детей 120 60,Матрасик \nПользовались 3-4 Месяца. \n120*60,1745944.jpg,3.00,30.0,330.0,174.0


In [92]:
train[train["real_weight"] > train["real_weight"].quantile(0.999)].sort_values(by="real_weight", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
8337,139419,2024-07-10,Б/у,5000.0,Транспорт,Запчасти и аксессуары,Электрооборудование,487099,514504,Проводка Peugeot 206 1.6,Проводка Peugeot 206 1.6 2007г\n\n трёхдверный...,139419.jpg,7090.0,35.0,50.0,40.0
45302,1413997,2024-07-13,Б/у,1500.0,Электроника,Аудио и видео,"Акустика, колонки, сабвуферы",1186113,1371196,Аудиосистема Microlab M-200вт,Аудиосистема Microlab M-200вт. В рабочем состо...,1413997.jpg,4555.0,30.0,35.0,35.0
244562,148210,2024-01-31,Новое,3100.0,Для дома и дачи,Мебель и интерьер,Постельное бельё,2004742,2324580,Постельное бельё с одеялом евро,🌺Постельное белье с одеялом и простынью на ре...,148210.jpg,3250.0,13.0,50.0,40.0
232180,1475743,2024-06-18,Новое,750.0,Для дома и дачи,Ремонт и строительство,Водоочистка и фильтры,1841643,1258887,Жидкость для биотуалета Тhetford,Жидкость Thetford для биотуалета. для верхнег...,1475743.jpg,3000.0,10.0,10.0,10.0
232709,458921,2024-02-07,Б/у,200.0,Личные вещи,Детская одежда и обувь,Комбинезоны,1961529,2217273,Одежда пакетом 68 размер (забронировали),Отдам одежду для девочки\nБесплатно при самовы...,458921.jpg,2740.0,30.0,40.0,40.0


#### Визуальный анализ показывает что не все объявления являются мусором, есть валидные крупногабаритные объекты

#### Проанализируем распределение таргетов по категориям. Для анализа будем использовать соотношение хвостов для квантилей 0.99 и 0.999

In [93]:
cat_col = "subcategory_name"
targets = ["real_height", "real_length", "real_width", "real_weight"]

q_table = (
    train
    .groupby(cat_col)[targets]
    .quantile([0.99, 0.999])
    .unstack(level=1)
)

bad_subcats = set()

for t in targets:
    ratio = q_table[(t, 0.999)] / q_table[(t, 0.99)]

    print(f"\nПодозрительное соотношение хвостов для {t}")
    display(
        ratio
        .sort_values(ascending=False)
        .head(20)
    )

    # собираем subcategory, где ratio > 2
    bad_subcats.update(
        ratio[ratio > 2].index
    )

bad_subcats = list(bad_subcats)
bad_subcats


Подозрительное соотношение хвостов для real_height


subcategory_name
Мебель и интерьер               1.827750
Детская одежда и обувь          1.666667
Игры, приставки и программы     1.650857
Одежда, обувь, аксессуары       1.612903
Книги и журналы                 1.525500
Товары для детей и игрушки      1.463415
Посуда и товары для кухни       1.463415
Фототехника                     1.458333
Аудио и видео                   1.453650
Коллекционирование              1.452314
Часы и украшения                1.448750
Телефоны                        1.447000
Красота и здоровье              1.428571
Оргтехника и расходники         1.395349
Товары для компьютера           1.388889
Музыкальные инструменты         1.377000
Велосипеды                      1.345022
Охота и рыбалка                 1.336541
Бытовая техника                 1.333333
Планшеты и электронные книги    1.333333
dtype: float64


Подозрительное соотношение хвостов для real_length


subcategory_name
Телефоны                       2.168077
Книги и журналы                2.158947
Ноутбуки                       2.090050
Одежда, обувь, аксессуары      1.950000
Спорт и отдых                  1.898108
Товары для детей и игрушки     1.831883
Детская одежда и обувь         1.779661
Ремонт и строительство         1.731551
Красота и здоровье             1.714286
Коллекционирование             1.685271
Игры, приставки и программы    1.683367
Оргтехника и расходники        1.668407
Фототехника                    1.628750
Часы и украшения               1.618414
Товары для компьютера          1.607343
Бытовая техника                1.583961
Посуда и товары для кухни      1.531046
Аудио и видео                  1.523340
Мебель и интерьер              1.484463
Запчасти и аксессуары          1.392924
dtype: float64


Подозрительное соотношение хвостов для real_width


subcategory_name
Спорт и отдых                  1.960784
Одежда, обувь, аксессуары      1.953300
Мебель и интерьер              1.851852
Посуда и товары для кухни      1.788462
Детская одежда и обувь         1.772727
Товары для детей и игрушки     1.719950
Фототехника                    1.716667
Коллекционирование             1.675054
Игры, приставки и программы    1.646978
Бытовая техника                1.636444
Ремонт и строительство         1.587120
Книги и журналы                1.585568
Запчасти и аксессуары          1.495769
Товары для компьютера          1.493257
Красота и здоровье             1.458333
Оргтехника и расходники        1.416827
Аудио и видео                  1.403509
Телефоны                       1.363636
Часы и украшения               1.363636
Ноутбуки                       1.250000
dtype: float64


Подозрительное соотношение хвостов для real_weight


subcategory_name
Планшеты и электронные книги    2.700437
Одежда, обувь, аксессуары       2.695104
Часы и украшения                2.679646
Детская одежда и обувь          2.654082
Телефоны                        2.583245
Красота и здоровье              1.943593
Фототехника                     1.931133
Книги и журналы                 1.903146
Охота и рыбалка                 1.761254
Посуда и товары для кухни       1.753279
Ноутбуки                        1.695045
Мебель и интерьер               1.694582
Товары для детей и игрушки      1.633333
Спорт и отдых                   1.601708
Товары для компьютера           1.581768
Коллекционирование              1.481144
Бытовая техника                 1.474014
Игры, приставки и программы     1.464316
Оргтехника и расходники         1.407916
Настольные компьютеры           1.366655
dtype: float64

['Часы и украшения',
 'Одежда, обувь, аксессуары',
 'Планшеты и электронные книги',
 'Детская одежда и обувь',
 'Книги и журналы',
 'Ноутбуки',
 'Телефоны']

In [94]:
N_EXAMPLES = 10

for subcat in bad_subcats:
    df_sub = train[train[cat_col] == subcat]
    print(f"SUBCATEGORY: {subcat}")

    for t in targets:
        q99 = df_sub[t].quantile(0.99)
        q999 = df_sub[t].quantile(0.999)

        ratio = q999 / q99

        if ratio > 2:
            # показываем хвостовые примеры (те, кто > q99)
            tail_examples = (
                df_sub[df_sub[t] > q99]
                .sort_values(t, ascending=False)
                .head(N_EXAMPLES)
            )

            display(
                tail_examples[["item_id", "title", "description", t]]
            )

SUBCATEGORY: Часы и украшения


,item_id,title,description,real_weight
259063,488655,Часы Seiko Monster,Часы Seiko Monster 4- го поколения.\nМеханизм ...,300.000
296611,681860,Часы (нерабочие) и корпуса часов на запчасти,Закрылась мастерская часов. Продается одним ло...,16.500
269219,1171793,Настенные часы с зеркалом,Большие настенные часы. \n\nРАБОЧИЕ! \n\nЧасы ...,15.008
201834,1656147,Смарт часы лотом,Смарт часы около 80 шт лотом,13.900
209191,231977,Часы с боем Янтарь раритет,"Продаю совдеповские часы производство СССР, с ...",13.630
245800,1540114,Коробки для часов Longines 037-12,‼️БРОНЬ \n\nАВИТО ДОСТАВКА,13.200
300992,1553241,Коробки для часов Longines В12-2,Коробки для часов Longines \n\n‼️Бронь,13.200
275126,1626267,"Часы настенные, ГДР, из натур. кожи",Раритетные в очень хорошем сохране.Из натурал...,12.600
43677,2043669,"83х83 см. Большие Часы СССР,дерево,ручная работа","Часы настенные,ручная работа, Белые капли-это ...",12.600
292558,1181764,Колеса на Мотороллер,"Б/у колеса на мотороллер, в хорошем состоянии....",11.150


SUBCATEGORY: Одежда, обувь, аксессуары


,item_id,title,description,real_weight
210680,1434798,Crocs женские на платформе,В ДОСТАВКЕ!\n\nСандалии crocs Crush Sandal\n\n...,1114.0
146828,1211458,Мужская кожаная сумка портфель Ritter,Продам мужскую сумку портфель из натуральной к...,1000.0
149641,766606,Кеды max mara 38,"Кеды белые кожаные Max Mara женские, 38 размер...",905.0
206220,316273,Ортопедическая обувь Мужская 39-46 размер,"Ортопедическая as обувь, мужская (39-46).\nНов...",865.0
150195,1404184,"Льняная юбка 44 р. S, синяя, Hobbs",Комфортная летом юбка от известного бренда.\n\...,796.0
137843,20590,Ботинки Raf Simons р.42,"Ботинки Raf Simons, в хорошем состоянии, стоил...",745.0
202081,266480,"Босоножки на каблуке, Zenden б/у 39 размер","1. Босоножки на каблуке, Zenden б/у - следы но...",700.0
196443,1483282,Пиджак женский Mango L,Продаю пиджак MANGO на двух пуговицах (застежк...,700.0
17635,1379021,Едет обратно. Твидовый пиджак женский укороченны,"Продам твидовый пиджак в отличном состоянии, б...",654.0
226036,410079,Винтажный свитер оверсайз Jaded london diesel ...,Очень клевая вещь! Интересный крой и приятные ...,645.0


SUBCATEGORY: Планшеты и электронные книги


...

SUBCATEGORY: Ноутбуки


,item_id,title,description,real_length
287110,821779,Ноутбук i5 3337/ gforce 740m (toshiba),"Строит новый ssd на 256 gb, windows 10. Работа...",1045.0
184179,1802697,Asus i7/12gb/ssd 240/Nvidia,Добро пожаловать в PRIME! \n\nНоутбук Asus в о...,140.0
257925,941861,"Ноутбук Irbis NB60, комплект, коробка, работает",Ноутбук в рабочем состоянии. От сети полностью...,138.0
255691,984368,Dell inspiron 7577 игровой,Хороший игровой ноутбук. Технически в идеально...,104.0
294617,812882,Коробки Macbook Pro Core i5 13 дюймов,Коробки от MacBook Pro Core i5 13 дюймов. Сост...,95.0
128501,1898705,"16"" huawei MateBook 16s cref-X i9-13900H",новый.\r\nна гарантии ДНС по всей РФ. \r\nполн...,83.0
184117,1639806,Ноутбук Lenovo Legion,Продается ноутбук Lenovo Legion + зарядное у...,80.0
13299,1160760,MSI GF63 RTX 3050/ I5 12450H / 16GB/ 144гц новый,"Игровой ноутбук, новый,запечатанный.\n\nБумажн...",80.0
189260,2092704,Asus Vivobook Pro 15 oled rtx3050,"ноутбук в очень хорошем состоянии, технически ...",80.0
292047,1015639,Ноутбук HP Pavilion DV6 7171er i7 ssd 256 8GB RAM,"Данный 15.6"" ноутбук с алюминиевой отделкой ко...",75.0


SUBCATEGORY: Телефоны


,item_id,title,description,real_length
23812,1694915,Радиостанция Intek M 790Plus,"Рация в рабочем состоянии, в комплекте антенна...",151.0
19700,1723908,"BQ 6035L Strike Power MAX, 2/32 ГБ",Разбили экран после пары месяцев как купили но...,150.0
155006,699049,Стационарная антенна для радиостанции,Антена для радиостанции си би. Длина 6м. Склад...,137.0
205523,1514237,Антенна radial A0 VHF,Бренд: РАДИАЛ 136-174 мгц VHF ✅\n\nКомплектаци...,120.0
970,1901920,Активная антенная система Yaesu atas-120А,Продаю автомобильную многодиапазонную (HF/VHF/...,105.0
207381,1670739,Базовая дипольная антенна D1 WHF Radial,Продам 4 петли на двухметровый диапазон. Прод...,83.0
106864,1006429,Антенна для рации anli новая,Anli AW-6 UHF Антенна автомобильная\nДиапазон ...,81.0
265828,1795740,Блок питания для радиостанции,Блок питания для радиостанции К-35 Alan.\n\nБл...,81.0
93858,1050205,Стойка для телефона,"Стойка для телефона, в разложенном состоянии 1...",80.0
191967,456303,"Fly IQ441 Radiance, 4 ГБ","Рабочий, но сенсор под замену! Аккумулятор раб...",78.0


,item_id,title,description,real_weight
99804,1867724,Коробка полная запчастей мобильных,коробка для алексея,24.750
139210,1439432,Планшеты много оптом рабочие и сломанные,"Планшеты - 1 ЛОТОМ ! ~ 200 шт. \nНа продажу, з...",24.500
299087,1733619,Кв трансивер UA1FA,Кв трансивер UA1FA пролежал 20 лет краска осы...,20.500
206734,698612,"Коробка с телефонами, электроникой и запчастями","Остаток от сервиса\nЕсть телефоны рабочие, нек...",16.000
171055,594608,"Телефон IP grandstream GXP2130, GXP2135",Продам БУ Телефоны IP GRANDSTREAM GXP2130-10шт...,15.000
266362,909239,500шт безпроводных наушников,Не рабочие на ремонт цена за все 500шт в налич...,15.000
159240,743971,Антенна для радиостанции Р163-10К,Комплект мачты с антенной для радиостанции Р16...,14.300
23812,1694915,Радиостанция Intek M 790Plus,"Рация в рабочем состоянии, в комплекте антенна...",13.197
166361,2014918,Армейский военно-полевой телефон та-57,Армейский военно-полевой телефонный аппарат ТА...,13.100
191695,1924695,"IP-телефоны Panasonic KX-NT511, Panasonic KX-N...",Комплект: \r\n6 IP-телефонов Panasonic KX-NT51...,12.032


#### Заметим, что иногда в объявлении продается много товаров, что существенно влияет на вес, можно использовать как доп фичу если в поле description есть ключевые слова: шт, все, комплект, коробка, чемодан, много, оптом

#### Удалим данные из категорий в которых объявления больше всего похожи на выбросы

In [95]:
# чистим "Детская одежда и обувь"

train = train.loc[
    ~(
        (train["subcategory_name"] == "Детская одежда и обувь") &
        (train["real_weight"] > 36)
    )
].copy()


In [96]:
# чистим "Одежда, обувь, аксессуары"

q99 = (
    train.loc[train["subcategory_name"] == "Одежда, обувь, аксессуары", "real_weight"]
    .quantile(0.99)
)

train = train.loc[
    ~(
        (train["subcategory_name"] == "Одежда, обувь, аксессуары") &
        (train["real_weight"] > q99)
    )
].copy()


In [97]:
# чистим "Книги и журналы"

q99 = (
    train.loc[train["subcategory_name"] == "Книги и журналы", "real_length"]
    .quantile(0.99)
)

train = train.loc[
    ~(
        (train["subcategory_name"] == "Книги и журналы") &
        (train["real_length"] > q99)
    )
].copy()

In [98]:
# чистим "Часы и украшения"

train = train.loc[
    ~(
        (train["subcategory_name"] == "Часы и украшения") &
        (train["real_weight"] > 17)
    )
].copy()

#### Снова посмотрим статистики таргета после чистки

In [99]:
train[targets].describe().T

,count,mean,std,min,25%,50%,75%,max
real_height,280772.0,11.574445,8.266541,1.000,6.000,10.00,15.00,288.0
real_length,280772.0,33.077362,214.871513,1.000,22.000,32.00,40.00,111363.0
real_width,280772.0,23.452185,23.380571,1.000,16.000,21.00,30.00,11029.0
real_weight,280772.0,1.613904,19.267445,0.001,0.339,0.69,1.48,7090.0


#### Судя по максимальных значениям выбросы все равно остались. Сделаем клип по квантили внутри подкатегорий

In [100]:
q_clip = 0.999

for t in targets:
    thr = train.groupby("subcategory_name")[t].transform(lambda s: s.quantile(q_clip)) # каждая строка получает потолок, соответствующий своей подкатегории.
    train[t] = np.minimum(train[t], thr) # поэлементный минимум


In [101]:
train[targets].describe().T

,count,mean,std,min,25%,50%,75%,max
real_height,280772.0,11.559187,8.151298,1.000,6.000,10.00,15.00,73.11000
real_length,280772.0,32.519969,16.241165,1.000,22.000,32.00,40.00,210.69000
real_width,280772.0,23.396771,10.607179,1.000,16.000,21.00,30.00,105.00000
real_weight,280772.0,1.514239,2.571149,0.001,0.339,0.69,1.48,30.42534


#### Выглядит адекватно, перейдем к анализу выбросов с точки зрения минимальных значений

In [102]:
train[targets].quantile([0.0, 0.01, 0.05]).T

,0.00,0.01,0.05
real_height,1.000,1.000,1.0
real_length,1.000,7.000,10.0
real_width,1.000,4.000,10.0
real_weight,0.001,0.018,0.1


In [103]:
sentinel_values = {
    "real_height": 1,
    "real_length": 1,
    "real_width": 1,
    "real_weight": 0.001,
}

for t, v in sentinel_values.items():
    print(t, (train[t] == v).mean())


real_height 0.12892667359993162
real_length 0.0012501246563047597
real_width 0.003910646360748222
real_weight 0.006172267889960537


#### Анализ показал, что почти 13 % объявлений имеют высоту 1 мм, скорее всего это заглушка и указана вместо реальной высоты. Такие значения могут сильно повредить регрессии, поэтому они будут исключены из обучения

In [104]:
train = train[train["real_height"] != 1].copy()

#### Проанализируем объявления у которых слишком малый вес, для этого посмотрим у каких категорий есть резкий скачок от нижних значений к нормальным

In [105]:
cat_col = "subcategory_name"
target = "real_weight"

q_table = (
    train
    .groupby(cat_col)[targets]
    .quantile([0.01, 0.05])
    .unstack(level=1)
)

bad_subcats = set()

for t in targets:
    ratio = q_table[(t, 0.05)] / q_table[(t, 0.01)]
    
    display(
        ratio
        .sort_values(ascending=False)
        .head(15)
    )

    bad_subcats.update(
        ratio[ratio > 10].index
    )

bad_subcats = list(bad_subcats)
bad_subcats


subcategory_name
Аудио и видео                   2.000000
Оргтехника и расходники         2.000000
Товары для компьютера           2.000000
Запчасти и аксессуары           2.000000
Музыкальные инструменты         2.000000
Охота и рыбалка                 2.000000
Настольные компьютеры           1.698113
Фототехника                     1.666667
Товары для детей и игрушки      1.666667
Спорт и отдых                   1.666667
Ремонт и строительство          1.666667
Мебель и интерьер               1.666667
Телефоны                        1.500000
Планшеты и электронные книги    1.500000
Часы и украшения                1.500000
dtype: float64

subcategory_name
Ноутбуки                       3.000000
Настольные компьютеры          1.750000
Детская одежда и обувь         1.600000
Мебель и интерьер              1.500000
Посуда и товары для кухни      1.500000
Телефоны                       1.428571
Часы и украшения               1.428571
Книги и журналы                1.400000
Бытовая техника                1.400000
Игры, приставки и программы    1.300000
Товары для детей и игрушки     1.300000
Спорт и отдых                  1.300000
Коллекционирование             1.250000
Оргтехника и расходники        1.200000
Товары для компьютера          1.100000
dtype: float64

subcategory_name
Ноутбуки                        2.000000
Запчасти и аксессуары           1.800000
Оргтехника и расходники         1.666667
Спорт и отдых                   1.666667
Посуда и товары для кухни       1.666667
Телефоны                        1.600000
Коллекционирование              1.600000
Красота и здоровье              1.600000
Охота и рыбалка                 1.600000
Товары для компьютера           1.500000
Планшеты и электронные книги    1.500000
Аудио и видео                   1.500000
Настольные компьютеры           1.500000
Музыкальные инструменты         1.500000
Велосипеды                      1.500000
dtype: float64

subcategory_name
Детская одежда и обувь          120.000000
Книги и журналы                 115.000000
Мебель и интерьер                90.000000
Одежда, обувь, аксессуары        80.000000
Посуда и товары для кухни        75.000000
Товары для детей и игрушки       50.000000
Ремонт и строительство           40.000000
Бытовая техника                  34.020000
Оргтехника и расходники          11.111111
Спорт и отдых                     6.750000
Планшеты и электронные книги      4.610951
Велосипеды                        4.247104
Коллекционирование                4.108885
Товары для компьютера             4.000000
Ноутбуки                          3.772932
dtype: float64

['Одежда, обувь, аксессуары',
 'Детская одежда и обувь',
 'Бытовая техника',
 'Книги и журналы',
 'Мебель и интерьер',
 'Ремонт и строительство',
 'Товары для детей и игрушки',
 'Оргтехника и расходники',
 'Посуда и товары для кухни']

#### Категории в которых отношение q05/q01 > 10 скорей всего содержат заглушки по весам, либо неправдоподоно мелкие веса

In [106]:
N_EXAMPLES = 10

for subcat in bad_subcats:
    df_sub = train[train[cat_col] == subcat]
    print(f"SUBCATEGORY: {subcat}")

    for t in targets:
        q01 = df_sub[t].quantile(0.01)
        q05 = df_sub[t].quantile(0.05)

        ratio = q05 / q01

        if ratio > 20:
            tail_examples = (
                df_sub[df_sub[t] < q05]
                .sort_values(t)
                .head(N_EXAMPLES)
            )

            display(
                tail_examples[["item_id", "title", "description", t]]
            )

SUBCATEGORY: Одежда, обувь, аксессуары


,item_id,title,description,real_weight
256266,1300998,Туфли женские новые 37 размер летние,Новые\nРазмер 37\nВ размер \n\nЛетняя модель т...,0.001
190456,745267,Оригинал mochino женские 38 и другое,"1) I love mochino оригинал 15000, 38 размер, о...",0.001
169538,1537772,Шапка ушанка форменная,Шапка ушанка форменная черная . Состояние про...,0.001
86835,152978,Туфли мужские 43 продам или обменяю,"Новые туфли,нат.кожа.Торг.",0.001
134430,1272964,"Жакет lime, размер s",Хлопковый жакет lime молочного цвета. Очень ег...,0.001
189346,1290291,Футболка Champion XL Vintage,Оригинальная винтажная футболка Чемпион. Цвет ...,0.001
60237,1744032,Кроссовки nike бу,"Кроссовки Nike SB Dunk Low ""The Beatles"" abbey...",0.001
215274,75914,Пуховик женский baon,"Уютный и тёплый пуховик, хорошее качество и со...",0.001
122570,1620570,Балетки натуральная кожа 39 Индия,"Балетки 39 размер, на узкую ножку, натуральная...",0.001
218720,94247,Босоножки. Каприз (немецкая фирма),"Босоножки каприз,6 1/2 на 38 очень удобные, ка...",0.001


SUBCATEGORY: Детская одежда и обувь


,item_id,title,description,real_weight
86218,1902195,Одежда для девочки пакетом р. 98-104,98/104р.\nполосатое -НМ длина 55 см\nОт подмы...,0.001
255663,1668779,Спортивный костюм Fila для девочки 146-152,Костюм бомбер и леггинсы. В отличном состоянии.,0.001
179655,1769567,Валенки филипок 27 для мальчика,Продам валенки филипок для мальчика 27 размера...,0.001
245699,1886803,Ботинки зимние натуральный мех 23р,"Ботинки зимние детские 23р-р \nОртопедические,...",0.001
61900,1003592,Конверт + комбинезон зимний для новорожденных,"Комбинезон+тёплый конверт, 0-3 мес. Описание и...",0.001
80540,253104,Джинсы на девочку 10 лет,Джинсы на девочку примерно 9-10 лет\nНизкая по...,0.001
268181,1302631,Парка зимняя куртка для мальчика,Куртка удлиненная на мальчика. Размер 134-140....,0.001
106549,545505,Зимний комбинезон 112,Отправлю посылку через пятерочку. \n\nКомбинез...,0.001
184959,1932871,Комбинезон зимний детский для девочкимальчика 74,Продам супер теплые зимний комбинезон для дево...,0.001
291951,1407199,Одежда в 1 класс,"3 кофты, сарафан синий. Все в отличном состоянии",0.001


SUBCATEGORY: Бытовая техника


...

,item_id,title,description,real_weight
197569,1122788,Газ лифты б/у,Газ лифты б/у. 200 за один газлифт. (остались...,0.001
281666,1028658,Шланг для мойки высокого давления 10 метров,Шланг высокого давления 10 метров \n\nГайка М2...,0.001
303386,169817,Гравёр Bosch GRO 12v-35,Аккумуляторный гравер Bosch GRO 12V-35 \n\nКом...,0.001
190351,57361,Шланг для полива magic hose + насадка,Шланг для полива MAGIC HOSE + насадка\n\nЦена ...,0.001
85949,1355259,Поливочный шланг новый,"Поливочный шланг 22,5 метра, в наличии 25 шт.",0.001
218753,706505,Линейный подшипник LM8UU,Новые 10 шт. Продаю так как пришли не того раз...,0.001
188767,767088,Шуруповëрт Dewalt dcd776,"Продаëтся шуруповёрт Деволт с функцией удара, ...",0.001
202075,410605,Перчатки спилковые сварочные зимние,Ераги спилковые сварочные утеплённые с флисово...,0.001
305744,1829328,Смеситель для раковины voda,"Пользовались ~5 месяцев, сняли по причине поку...",0.001
132426,1119494,Детектор напряжения FIT IT 56530,FIT 56530\n\nДетектор напряжения\n\nДетектор н...,0.001


SUBCATEGORY: Товары для детей и игрушки


,item_id,title,description,real_weight
252579,1078302,Bondibon SMART games День и ночь оригинал,"Полный комплект - все фигуры, инструкция, карт...",0.001
168413,1309451,Игра Тир проекционный Джони черепок,"Интересная, захватывающая игра. Всё работает.",0.001
152215,1806006,Молокоотсос электрический двойной,Молокоотсос электрический двойной.\nСостояние ...,0.001
132819,1602833,Бизикуб и сенсорная игрушка,"Бизикуб пингвин развивающий, очень удобно брат...",0.001
251893,1728803,"Конструктор ""Полесье""","Продам конструктор ""Полесье"" в хорошем состоян...",0.001
111834,270795,Пазл карта мира Castorland maxi,Maxi пазл Castorland «Карта мира»\nОдна деталь...,0.001
118062,1677253,Стерилизатор - подогреватель,Стерилизатор - подогреватель в хорошем состоян...,0.001
80366,2077539,Lego juniors Тачки,Отличный большой набор Лего Тачки (оригинал). ...,0.001
109805,232619,Лежак детский с антимоскитной сеткой,"Очень удобный лежак, можно ходить в гости, на ...",0.001
94123,1675391,Подгузники трусики nao xl,"открытая пачка трусиков подгузников nao XL(5),...",0.001


SUBCATEGORY: Оргтехника и расходники
SUBCATEGORY: Посуда и товары для кухни


,item_id,title,description,real_weight
61670,496783,Средство для очистки Nespresso кофе машины,Средство. Новое.,0.001
231087,790269,Ковшик антипригарный scovo 16 см,Ковшик с антипригарным покрытием scovo consul ...,0.001
888,2049875,Вилки нержавейка СССР 8 штук,Вилки ссср нержавейка 8 шт,0.001
122869,481032,"Посуда СССР фарфор, стекло, керамика",Посуда СССР в отличном состоянии\nБольшой зава...,0.001
266978,1179810,Лёдница со щипцами Jameson Ирландия,Емкость для льда Jameson Irish Whiskey новая. ...,0.001
103675,956631,Чайник слоник салатник конфетница,Два великолепных изделия ручной работы. Чайник...,0.001
9095,233805,"Полка для барной стойки центральная D350 мм, хром","ПОЛКА ДЛЯ БАРНОЙ СТОЙКИ ЦЕНТРАЛЬНАЯ D350 ММ, Х...",0.001
96854,1890956,Бронь для Serj K,бутылки для соус,0.001
236002,854832,Кофе в капсулах для кофемашины,продаю две банки сразу - 300 руб самовывоз \nс...,0.001
8898,1870193,Гейзерная кофеварка индукция 450мл,"Покупали в подарок, подарить не получилось.\n\...",0.001


#### Удалим только минимальные значения weight = 0.001

In [107]:
train = train[train["real_weight"] != 0.001].copy()

### 3. Добавление новых фичей

In [ ]:
train.info()

In [ ]:
feat_extra_cat = []
feat_extra_num = []

In [ ]:
# объединим в одну фичу title + description

def add_text_join(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text_join"] = (
        df["title"] + " " +
        df["description"].fillna("")
    )
    return df

train = add_text_join(train)
val = add_text_join(val)
test = add_text_join(test)

In [ ]:
# Добавим фичу доля заглавных букв в title
def caps_ratio(s: str) -> float:
    """доля заглавных букв в тексте,
    рассчитанная как отношение количества
    заглавных букв к общему количеству букв"""
    if not s:
        return 0.0

    letters = sum(ch.isalpha() for ch in s)
    if letters == 0:
        return 0.0

    caps = sum(ch.isupper() for ch in s)
    return caps / letters

train["caps_ratio"] = train["title"].apply(lambda x: caps_ratio(x))
val["caps_ratio"] = val["title"].apply(lambda x: caps_ratio(x))
test["caps_ratio"] = test["title"].apply(lambda x: caps_ratio(x))

feat_extra_num.append("caps_ratio")

In [ ]:
# Добавим фичу доля букв и цифр в title 

def alnum_ratio(s: str) -> float:
    """доля букв и цифр в тексте,
    рассчитанная как отношение количества
    букв и цифр к общему количеству символов с пробелами"""
    if not s:
        return 0.0

    total = len(s)
    if total == 0:
        return 0.0

    alnum = sum(ch.isalnum() for ch in s)
    return alnum / total

train["alnum_ratio"] = train["title"].apply(lambda x: alnum_ratio(x))
val["alnum_ratio"] = val["title"].apply(lambda x: alnum_ratio(x))
test["alnum_ratio"] = test["title"].apply(lambda x: alnum_ratio(x))

feat_extra_num.append("alnum_ratio")

In [ ]:
# добавим фичу количество символов в title
train["text_len"] = train["title"].apply(lambda x: len(x))
val["text_len"] = val["title"].apply(lambda x: len(x))
test["text_len"] = test["title"].apply(lambda x: len(x))

feat_extra_num.append("text_len")

In [ ]:
# log количество слов title
train["log_words_cnt"] = train["title"].apply(lambda x: np.log1p(len(x.split())))
val["log_words_cnt"] = val["title"].apply(lambda x: np.log1p(len(x.split())))
test["log_words_cnt"] = test["title"].apply(lambda x: np.log1p(len(x.split())))

feat_extra_num.append("log_words_cnt")

In [ ]:
# сделаем фичу - флаг, что в объявлении множество товаров

pattern = re.compile(
    r"(?:\bшт\b|\d+\s*шт|\bштук\b|\bкомплект\b|\bчемодан\b|\bнабор\b|\bоптом\b)",
    flags=re.IGNORECASE
)

train["has_bundle_words"] = (
    train["text_join"]
    .fillna("")
    .str.lower()
    .apply(lambda x: int(bool(pattern.search(x))))
)

val["has_bundle_words"] = (
    val["text_join"]
    .fillna("")
    .str.lower()
    .apply(lambda x: int(bool(pattern.search(x))))
)

test["has_bundle_words"] = (
    test["text_join"]
    .fillna("")
    .str.lower()
    .apply(lambda x: int(bool(pattern.search(x))))
)
 
feat_extra_cat.append("has_bundle_words")

In [ ]:
# сделаем фичу - флаг, что в объявлении есть единицы измерения

train["has_size"] = train["text_join"].str.contains(
    r"(?:\bмм\b|\bсм\b|\bм\b|\bметр(?:а|ов)?\b|\d+(?:[.,]\d+)?\s*(?:мм|см|м)\b)",
    case=False,
    regex=True
).astype(int)

val["has_size"] = val["text_join"].str.contains(
    r"(?:\bмм\b|\bсм\b|\bм\b|\bметр(?:а|ов)?\b|\d+(?:[.,]\d+)?\s*(?:мм|см|м)\b)",
    case=False,
    regex=True
).astype(int)

test["has_size"] = test["text_join"].str.contains(
    r"(?:\bмм\b|\bсм\b|\bм\b|\bметр(?:а|ов)?\b|\d+(?:[.,]\d+)?\s*(?:мм|см|м)\b)",
    case=False,
    regex=True
).astype(int)

feat_extra_cat.append("has_size")

In [ ]:
# добавим фичи месяц заказа order_month и день недели order_dow
dt_train = pd.to_datetime(train["order_date"], errors="coerce")
train["order_month"] = dt_train.dt.month.astype("Int64")
train["order_dow"] = dt_train.dt.dayofweek.astype("Int64") 

dt_val = pd.to_datetime(val["order_date"], errors="coerce")
val["order_month"] = dt_val.dt.month.astype("Int64")
val["order_dow"] = dt_val.dt.dayofweek.astype("Int64") 

dt_test = pd.to_datetime(test["order_date"], errors="coerce")
test["order_month"] = dt_test.dt.month.astype("Int64")
test["order_dow"] = dt_test.dt.dayofweek.astype("Int64") 

feat_extra_cat.append("order_month")
feat_extra_cat.append("order_dow")


In [ ]:
# добавим фичу log_item_price
train["log_item_price"] = np.log1p(train["item_price"])
val["log_item_price"] = np.log1p(val["item_price"])
test["log_item_price"] = np.log1p(test["item_price"])

feat_extra_num.append("log_item_price")

### 4. Генерация мета фичей на out of folds preds (OOF)

#### 4.1 Ridge на char-tf-idf векторах title clean

#### 4.1.1 Очистка title для tf-idf

In [ ]:
russian_stopwords = set(stopwords.words("russian"))

def simple_clean(s: str) -> str:
    """
    Функция для простой чистки текстов
    """
    if not s:
        return ""
    
    s = s.lower().replace("ё", "е")
    s = re.sub(r"[^a-zа-я0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    toks = s.split(" ")
    out = []

    for t in toks:
        if len(t) < 2 or t in russian_stopwords:
            continue
        out.append(t)

    return " ".join(out)

train["title_clean"] = train["title"].apply(lambda x: simple_clean(x))
train["text_join_clean"] = train["text_join"].apply(lambda x: simple_clean(x))

val["title_clean"] = val["title"].apply(lambda x: simple_clean(x))
val["text_join_clean"] = val["text_join"].apply(lambda x: simple_clean(x))

test["title_clean"] = test["title"].apply(lambda x: simple_clean(x))
test["text_join_clean"] = test["text_join"].apply(lambda x: simple_clean(x))


#### 4.1.2 Сделаем OOF предсказания таргетов на title_clean char-tf-idf векторах

In [ ]:
targets = ["real_height", "real_length", "real_width", "real_weight"]

def oof_ridge_tfidf_multi_log(
    train_df: pd.DataFrame,
    holdout_df: pd.DataFrame,
    test_df: pd.DataFrame,
    text_col: str,
    targets: list[str],
    n_splits: int = 5,
    seed: int = 42,
    # tfidf params
    analyser="word",
    ngram_range=(1, 2),
    min_df: int = 2,
    max_features: int = 200_000,
    sublinear_tf: bool = True,
    # ridge param
    alpha: float = 10.0,
):  
    X_text = train_df[text_col].values
    X_text_test = test_df[text_col].values
    X_text_hold = holdout_df[text_col].values

    # словари для хранения OOF_preds по каждому таргету
    oof_log = {t: np.zeros(len(train_df), dtype=np.float32) for t in targets}
    test_log_sum = {t: np.zeros(len(test_df), dtype=np.float32) for t in targets}
    hold_log_sum = {t: np.zeros(len(holdout_df), dtype=np.float32) for t in targets}

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # обучение по фолдам
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_text), 1):
        X_tr_raw = X_text[tr_idx]
        X_val_raw = X_text[val_idx]

        vec = TfidfVectorizer(
            analyzer=analyser,
            ngram_range=ngram_range,
            min_df=min_df,
            max_features=max_features,
            sublinear_tf=sublinear_tf, # заменяет tf на 1 + log(tf)
        )
        X_tr = vec.fit_transform(X_tr_raw)
        X_val = vec.transform(X_val_raw)
        X_test = vec.transform(X_text_test)
        X_h = vec.transform(X_text_hold)

        for t in targets:
            y = train_df[t].values
            y_tr_log = np.log1p(y[tr_idx])

            model = Ridge(alpha=alpha)
            model.fit(X_tr, y_tr_log)

            # val fold oof preds
            oof_log[t][val_idx] = model.predict(X_val).astype(np.float32)

            # holdout df preds (среднее по фолдам)
            hold_log_sum[t] += model.predict(X_h).astype(np.float32)

            # test preds (среднее по фолдам)
            test_log_sum[t] += model.predict(X_test).astype(np.float32)
        
        print(f"fold {fold}/{n_splits} done")

    # OOF dataframe
    oof_df = pd.DataFrame(
        {f"oof_ridge_{analyser}_{text_col}_{t}_log": oof_log[t] for t in targets},
        index=train_df.index
    )

    holdout_pred_df = pd.DataFrame(
        {f"oof_ridge_{analyser}_{text_col}_{t}_log": (hold_log_sum[t] / n_splits) for t in targets},
        index=holdout_df.index
    )

    test_pred_df = pd.DataFrame(
        {f"oof_ridge_{analyser}_{text_col}_{t}_log": (test_log_sum[t] / n_splits) for t in targets},
        index=test_df.index
    )

    # log MAE
    scores = []
    for t in targets:
        y_true_log = np.log1p(train_df[t].values)
        y_pred_log = oof_log[t]
        mae_log = mean_absolute_error(y_true_log, y_pred_log)
        scores.append({"target": t, "mae_log": mae_log})
    
    scores_df = pd.DataFrame(scores)
    scores_df.loc["mean"] = ["mean", scores_df["mae_log"].mean()]

    return oof_df, holdout_pred_df, test_pred_df, scores_df


train_oof_char_title_log, val_oof_char_title_log, test_oof_char_title_log, scores_char = oof_ridge_tfidf_multi_log(
    train_df=train,
    holdout_df=val,
    test_df=test,
    text_col="title_clean",
    targets=targets,
    n_splits=5,
    seed=42,
    analyser="char",
    ngram_range=(2, 4),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    alpha=10.0,
)

train = pd.concat([train, train_oof_char_title_log], axis=1).copy()
val = pd.concat([val, val_oof_char_title_log], axis=1).copy()
test = pd.concat([test, test_oof_char_title_log], axis=1).copy()

display(scores_char)
train_oof_char_title_log.head()

In [ ]:
# сохраним названия фичей
feat_oof_ridge_tfidf_title_clean = train_oof_char_title_log.columns.tolist()

#### 4.1.3 Сгенерим еще extra фичи на основе ridge char title clean oof preds log

In [ ]:
# сделаем отдельный список для названий экстра мета фичей
feat_meta_proxy = []

In [ ]:
# log отношения размеров
# oof_len_log - oof_height_log
train["oof_ridge_char_title_clean_log_ratio_l_h"] = train["oof_ridge_char_title_clean_real_length_log"] - train["oof_ridge_char_title_clean_real_height_log"]
val["oof_ridge_char_title_clean_log_ratio_l_h"] = val["oof_ridge_char_title_clean_real_length_log"] - val["oof_ridge_char_title_clean_real_height_log"]
test["oof_ridge_char_title_clean_log_ratio_l_h"] = test["oof_ridge_char_title_clean_real_length_log"] - test["oof_ridge_char_title_clean_real_height_log"]

# #oof_width_log - oof_height_log
train["oof_ridge_char_title_clean_log_ratio_w_h"] = train["oof_ridge_char_title_clean_real_width_log"] - train["oof_ridge_char_title_clean_real_height_log"]
val["oof_ridge_char_title_clean_log_ratio_w_h"] = val["oof_ridge_char_title_clean_real_width_log"] - val["oof_ridge_char_title_clean_real_height_log"]
test["oof_ridge_char_title_clean_log_ratio_w_h"] = test["oof_ridge_char_title_clean_real_width_log"] - test["oof_ridge_char_title_clean_real_height_log"]

In [ ]:
feat_meta_proxy.append("oof_ridge_char_title_clean_log_ratio_l_h")
feat_meta_proxy.append("oof_ridge_char_title_clean_log_ratio_w_h")

In [ ]:
# порядок величин
train["oof_ridge_char_title_clean_is_len_max"] = (
    train["oof_ridge_char_title_clean_real_length_log"]
    > train[[
        "oof_ridge_char_title_clean_real_width_log",
        "oof_ridge_char_title_clean_real_height_log"
    ]].max(axis=1)
).astype(int)

val["oof_ridge_char_title_clean_is_len_max"] = (
    val["oof_ridge_char_title_clean_real_length_log"]
    > val[[
        "oof_ridge_char_title_clean_real_width_log",
        "oof_ridge_char_title_clean_real_height_log"
    ]].max(axis=1)
).astype(int)

test["oof_ridge_char_title_clean_is_len_max"] = (
    test["oof_ridge_char_title_clean_real_length_log"]
    > test[[
        "oof_ridge_char_title_clean_real_width_log",
        "oof_ridge_char_title_clean_real_height_log"
    ]].max(axis=1)
).astype(int)

In [ ]:
feat_meta_proxy.append("oof_ridge_char_title_clean_is_len_max")

In [ ]:
# стандарное отклонение
train["oof_ridge_char_title_clean_spread_log"] = train[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log"
]].std(axis=1)

val["oof_ridge_char_title_clean_spread_log"] = val[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log"
]].std(axis=1)

test["oof_ridge_char_title_clean_spread_log"] = test[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log"
]].std(axis=1)

In [ ]:
feat_meta_proxy.append("oof_ridge_char_title_clean_spread_log")

In [ ]:
# proxy объема
# oof_volume_log ≈ oof_h_log + oof_w_log + oof_l_log
train["oof_ridge_char_title_clean_volume_log"] = train[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log",
]].sum(axis=1)

val["oof_ridge_char_title_clean_volume_log"] = val[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log",
]].sum(axis=1)

test["oof_ridge_char_title_clean_volume_log"] = test[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log",
]].sum(axis=1)

In [ ]:
feat_meta_proxy.append("oof_ridge_char_title_clean_volume_log")

In [ ]:
# proxy плотности
train["oof_ridge_char_title_clean_log_density"] = (
    train["oof_ridge_char_title_clean_real_weight_log"]
    - train["oof_ridge_char_title_clean_volume_log"]
)

val["oof_ridge_char_title_clean_log_density"] = (
    val["oof_ridge_char_title_clean_real_weight_log"]
    - val["oof_ridge_char_title_clean_volume_log"]
)

test["oof_ridge_char_title_clean_log_density"] = (
    test["oof_ridge_char_title_clean_real_weight_log"]
    - test["oof_ridge_char_title_clean_volume_log"]
)

In [ ]:
feat_meta_proxy.append("oof_ridge_char_title_clean_log_density")

#### 4.2 Ridge на word-tf-idf векторах title_clean

In [ ]:
train_oof_word_title_log, val_oof_word_title_log, test_oof_word_title_log, scores_word = oof_ridge_tfidf_multi_log(
    train_df=train,
    holdout_df=val,
    test_df=test,
    text_col="title_clean",
    targets=targets,
    n_splits=5,
    seed=42,
    analyser="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    alpha=10.0,
)

train = pd.concat([train, train_oof_word_title_log], axis=1)
val = pd.concat([val, val_oof_word_title_log], axis=1)
test = pd.concat([test, test_oof_word_title_log], axis=1)

display(scores_word)
train_oof_word_title_log.head()

In [ ]:
feat_oof_ridge_word_title_clean = train_oof_word_title_log.columns.tolist()

#### 4.3 Ridge на word-tf-idf векторах text_join_clean (title + description)

In [ ]:
train_oof_word_text_join_log, val_oof_word_text_join_log, test_oof_word_text_join_log, scores_word_text_join = oof_ridge_tfidf_multi_log(
    train_df=train,
    holdout_df=val,
    test_df=test,
    text_col="text_join_clean",
    targets=targets,
    n_splits=5,
    seed=42,
    analyser="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    alpha=10.0,
)

train = pd.concat([train, train_oof_word_text_join_log], axis=1)
val = pd.concat([val, val_oof_word_text_join_log], axis=1)
test = pd.concat([test, test_oof_word_text_join_log], axis=1)

display(scores_word_text_join)
train_oof_word_text_join_log.head()

In [ ]:
feat_oof_ridge_word_text_join = train_oof_word_text_join_log.columns.to_list()

#### 4.4 Ridge на RuBert эмбеддингах text_join (title + description)

In [ ]:
# чтение данных
train_rubert_emb = pd.read_parquet("RuBert_train.parquet")
test_rubert_emb = pd.read_parquet("RuBert_test.parquet")

In [ ]:
sub_tr = train[["item_id"] + targets]
sub_v = val[["item_id"] + targets]

train_rubert = pd.merge(sub_tr, train_rubert_emb, how="left", on="item_id").reset_index(drop=True)
val_rubert = pd.merge(sub_v, train_rubert_emb, how="left", on="item_id").reset_index(drop=True)
test_rubert = test_rubert_emb.reset_index(drop=True)

# приведем в соответствие индексы
train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

rubert_emb_cols = train_rubert_emb.columns[1:]

In [ ]:
def oof_ridge_on_emb_multi_log(
    emb_name: str,
    train_df: pd.DataFrame,
    holdout_df: pd.DataFrame,
    test_df: pd.DataFrame,
    emb_cols: list[str],
    targets: list[str],
    n_splits: int = 5,
    seed: int = 42,
    alpha: float = 10.0,
):
    X_train = train_df[emb_cols].values
    X_hold  = holdout_df[emb_cols].values
    X_test  = test_df[emb_cols].values

    # OOF / holdout / test
    oof_log = {t: np.zeros(len(train_df), dtype=np.float32) for t in targets}
    hold_log_sum = {t: np.zeros(len(holdout_df), dtype=np.float32) for t in targets}
    test_log_sum = {t: np.zeros(len(test_df), dtype=np.float32) for t in targets}

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr_raw = X_train[tr_idx]
        X_val_raw = X_train[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr_raw)
        X_val = scaler.transform(X_val_raw)
        X_h   = scaler.transform(X_hold)
        X_te  = scaler.transform(X_test)

        for t in targets:
            y = train_df[t].values
            y_tr_log = np.log1p(y[tr_idx])

            model = Ridge(alpha=alpha)
            model.fit(X_tr, y_tr_log)

            # OOF (train)
            oof_log[t][val_idx] = model.predict(X_val).astype(np.float32)

            # holdout (val) - среднее по фолдам
            hold_log_sum[t] += model.predict(X_h).astype(np.float32)

            # test - среднее по фолдам
            test_log_sum[t] += model.predict(X_te).astype(np.float32)

        print(f"fold {fold}/{n_splits} done")

    # OOF dataframe
    oof_df = pd.DataFrame(
        {f"oof_ridge_{emb_name}_{t}_log": oof_log[t] for t in targets},
        index=train_df.index,
    )

    holdout_pred_df = pd.DataFrame(
        {f"oof_ridge_{emb_name}_{t}_log": (hold_log_sum[t] / n_splits) for t in targets},
        index=holdout_df.index,
    )

    test_pred_df = pd.DataFrame(
        {f"oof_ridge_{emb_name}_{t}_log": (test_log_sum[t] / n_splits) for t in targets},
        index=test_df.index,
    )

    # log MAE
    scores = []
    for t in targets:
        y_true_log = np.log1p(train_df[t].values)
        y_pred_log = oof_log[t]
        mae_log = mean_absolute_error(y_true_log, y_pred_log)
        scores.append({"target": t, "mae_log": mae_log})

    scores_df = pd.DataFrame(scores)
    scores_df.loc["mean"] = ["mean", scores_df["mae_log"].mean()]

    return oof_df, holdout_pred_df, test_pred_df, scores_df


train_oof_rubert_emb_log, val_oof_rubert_emb_log, test_oof_rubert_emb_log, scores_rubert = (
    oof_ridge_on_emb_multi_log(
        emb_name="rubert",
        train_df=train_rubert,
        holdout_df=val_rubert,
        test_df=test_rubert,
        emb_cols=rubert_emb_cols.tolist(),
        targets=targets,
        n_splits=5,
        seed=42,
        alpha=10.0,
    )
)

train = pd.concat([train, train_oof_rubert_emb_log], axis=1).copy()
val   = pd.concat([val,   val_oof_rubert_emb_log],   axis=1).copy()
test  = pd.concat([test,  test_oof_rubert_emb_log],  axis=1).copy()

display(scores_rubert)
train_oof_rubert_emb_log.head()

In [ ]:
feat_oof_ridge_rubert_text_join = train_oof_rubert_emb_log.columns.to_list()

#### 4.5 Ridge на Dino-v2 эмбеддингах фотографий

In [ ]:
# чтение данных

train_dino_emb = pd.read_parquet("dinov2_train.parquet")
test_dino_emb = pd.read_parquet("dinov2_test.parquet")

In [ ]:
dino_emb_cols = train_dino_emb.columns[1:]

In [ ]:
sub_tr = train[targets + ["image_name"]]
sub_v = val[targets + ["image_name"]]
train_dino = pd.merge(sub_tr, train_dino_emb, how="left", on="image_name")
val_dino = pd.merge(sub_v, train_dino_emb, how="left", on="image_name")

In [ ]:
train_oof_dino_emb_log, val_oof_dino_emb_log, test_oof_dino_emb_log, scores_dino = oof_ridge_on_emb_multi_log(
    emb_name="dino",
    train_df=train_dino,
    holdout_df=val_dino,
    test_df=test_dino_emb,
    emb_cols=dino_emb_cols.tolist(),
    targets=targets,
    n_splits=5,
    seed=42,
    alpha=10.0,
)

train = pd.concat([train, train_oof_dino_emb_log], axis=1).copy()
val = pd.concat([val, val_oof_dino_emb_log], axis=1).copy()
test = pd.concat([test, test_oof_dino_emb_log], axis=1).copy()

display(scores_dino)
train_oof_dino_emb_log.head()

In [ ]:
feat_oof_ridge_dino = train_oof_dino_emb_log.columns.to_list()

### 5. Обучение итоговой GB модели

In [ ]:
# списки признаков для модели
cat_features = ["item_condition", "category_name",
                "subcategory_name", "microcat_name"] + feat_extra_cat

meta_features = (feat_oof_ridge_tfidf_title_clean
                 + feat_oof_ridge_word_text_join
                 + feat_oof_ridge_word_title_clean
                 + feat_oof_ridge_rubert_text_join
                 + feat_oof_ridge_dino
                 + feat_meta_proxy) 

feature_cols = cat_features + meta_features + feat_extra_num

targets = [
    "real_weight",
    "real_height",
    "real_width",
    "real_length"
]

In [ ]:
# проверка что все фичи не содержат Nan
assert train[feature_cols].isna().sum().sum() == 0
assert val[feature_cols].isna().sum().sum() == 0
assert test[feature_cols].isna().sum().sum() == 0

#### 5.1 Подберем гиперпараметры для итоговой модели, используя random search

In [ ]:
def eval_params(
        params: dict,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        feature_cols: list[str],
        cat_features: list[str],
        targets: list[str]
) -> float:
    """
    Функция для оценки параметров CatBoostRegressor
    """
    scores = []
    for t in targets:
        train_pool = Pool(
            train_df[feature_cols],
            label=np.log1p(train_df[t]),
            cat_features=cat_features,
        )
        val_pool = Pool(
            val_df[feature_cols],
            label=np.log1p(val_df[t]),
            cat_features=cat_features,
        )

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)

        z_pred = model.predict(val_pool)
        z_true = np.log1p(val_df[t]).to_numpy()
        scores.append(np.mean(np.abs(z_true - z_pred)))

    return float(np.mean(scores))


def sample_params(rng: np.random.Generator) -> dict:
    """
    Рандомный сэмплер гиперпараметров для CatBoost
    """
    # глубина деревьев
    depth = int(rng.integers(6, 11)) # 6..10

    # логарифмический сэмплинг lr
    lr = float(10 ** rng.uniform(-2.2, -1.0)) # ~0.006..0.1

    # L2-регуляризация листьев
    l2 = float(10 ** rng.uniform(0.0, 1.3)) # ~1..20

    # доля признаков, используемых при построении дерева
    rsm = float(rng.uniform(0.6, 1.0))

    # минимальное число объектов в листе
    min_leaf = int(rng.integers(1, 80))

    if rng.random() < 0.5:
        params = dict(
            bootstrap_type="Bernoulli",
            subsample=float(rng.uniform(0.6, 1.0)),
        )
    else:
        params = dict(
            bootstrap_type="Bayesian",
            bagging_temperature=float(rng.uniform(0.0, 1.0)),
        )

    params.update(dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=1000,
        learning_rate=lr,
        depth=depth,
        l2_leaf_reg=l2,
        rsm=rsm,
        min_data_in_leaf=min_leaf,
        random_strength=float(rng.uniform(0.0, 2.0)), # коэф. шума, который добавляется к оценке качества сплитов при построении дерева.
        random_seed=42,
        od_type="Iter",
        od_wait=300,
        verbose=False,
        thread_count=4,
    ))

    return params


# Random search
rng = np.random.default_rng(42)

N_TRIALS = 40
best_score = 1e9
best_params = None

for i in trange(N_TRIALS, desc="Random search", unit="trial"):
    params_i = sample_params(rng)
    sc = eval_params(
        params_i,
        train,
        val,
        feature_cols,
        cat_features,
        targets,
    )

    if sc < best_score:
        best_score, best_params = sc, params_i
        print(f"\n[{i+1}/{N_TRIALS}] new best score={best_score:.6f}")
    else:
        print(f"\n[{i+1}/{N_TRIALS}] score={sc:.6f}")

print("BEST:", best_score)
print(best_params)


#### 5.2 Итоговое обучение модели на всем train на лучших параметрах из random search

In [ ]:
best_params = {'bootstrap_type': 'Bernoulli',
        'subsample': 0.8607724102319897,
        'loss_function': 'MAE',
        'eval_metric': 'MAE',
        'iterations': 1000,
        'learning_rate': 0.033398625096323666,
        'depth': 10,
        'l2_leaf_reg': 3.438294242829107,
        'rsm': 0.7496736173628731,
        'min_data_in_leaf': 69,
        'random_strength': 1.7349812635046498,
        'random_seed': 42,
        'od_type': 'Iter',
        'od_wait': 300,
        'verbose': False,
        'thread_count': 4}

In [ ]:
params = best_params.copy()
params.update(dict(
        iterations=1600,
        verbose=200,
        thread_count=-1
))
params.pop("bootstrap_type", None)

models = {}
scores = {}

for t in targets:

    train_pool = Pool(
        data=train[feature_cols],
        label=np.log1p(train[t]),
        cat_features=cat_features,
    )

    val_pool = Pool(
        data=val[feature_cols],
        label=np.log1p(val[t]),
        cat_features=cat_features,
    )

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    z_pred = model.predict(val_pool)
    z_true = np.log1p(val[t]).to_numpy()
    log_mae = float(np.mean(np.abs(z_true - z_pred)))

    models[t] = model
    scores[t] = log_mae

    print(f"{t}: Val Log-MAE = {log_mae:.6f} | bestIter = {model.get_best_iteration()}")

score = float(np.mean(list(scores.values())))
print(f"score = {score}")

#### Самая большая ошибка на val датасете на real_height, вероятно дело в заглушках. Обучим иттоговую модель на всем трейне

In [ ]:
full_train = pd.concat([train, val], axis=0)

In [ ]:
models = {}

for t in targets:
    train_pool = Pool(
        data=full_train[feature_cols],
        label=np.log1p(full_train[t]),
        cat_features=cat_features,
    )
    
    params_final = params.copy()
    params_final.pop("od_type", None)
    params_final.pop("od_wait", None)

    model = CatBoostRegressor(**params_final)
    model.fit(train_pool, verbose=200)

    models[t] = model

print("Final models trained on full train")

In [ ]:
# pool для теста
test_pool = Pool(
    data=test[feature_cols],
    cat_features=cat_features,
)

# предсказания
preds = {}
for t in targets:
    z_pred = models[t].predict(test_pool)
    y_pred = np.expm1(z_pred)
    preds[t] = y_pred.astype(float)

submission = pd.DataFrame({
    "item_id": test["item_id"].values,
    "weight": preds["real_weight"],
    "height": preds["real_height"],
    "length": preds["real_length"],
    "width": preds["real_width"]
})

submission.to_csv("submission.csv", index=False)